# Healthcare Analytics Demo — One-Click Launcher

**Run All** to deploy the complete Healthcare Payer/Provider analytics solution into this Fabric workspace.

### What Gets Deployed
| Layer | Items |
|-------|-------|
| **Lakehouses** | `lh_bronze_raw`, `lh_silver_stage`, `lh_silver_ods`, `lh_gold_curated` |
| **Notebooks** | 5 ETL + 2 data generation |
| **Pipelines** | `PL_Healthcare_Full_Load`, `PL_Healthcare_Master` |
| **Semantic Model** | `HealthcareDemoHLS` (star schema) |
| **Data Agent** | `HealthcareHLSAgent` (Copilot AI) |
| **Ontology** | `Healthcare_Demo_Ontology_HLS` (via REST API) |

### Prerequisites
- An **empty** Fabric workspace (this notebook should be the only item)
- Fabric capacity: **F64** or higher recommended
- User must be workspace **Admin** or **Member**

### Configuration
Edit the cell below to point to your GitHub repo (public or private).

In [ ]:
# ============================================================================
# CONFIGURATION — Edit these values
# ============================================================================

GITHUB_OWNER = "kwame-one"           # GitHub org or user
GITHUB_REPO  = "Fabric-Payer-Provider-HealthCare-Demo"
GITHUB_BRANCH = "main"
GITHUB_TOKEN = ""                    # Leave empty for public repos

# Deployment options
GENERATE_DATA = True                  # Generate fresh synthetic data
RUN_PIPELINE = True                   # Run the full-load pipeline after data gen
DEPLOY_ONTOLOGY = True                # Deploy the GraphQL ontology (preview)
UPLOAD_KNOWLEDGE_DOCS = True          # Upload healthcare knowledge docs to lakehouse

print(f"Will deploy from: github.com/{GITHUB_OWNER}/{GITHUB_REPO} @ {GITHUB_BRANCH}")

In [ ]:
# ============================================================================
# CELL 1 — Install fabric-launcher
# ============================================================================

%pip install fabric-launcher --quiet

import notebookutils
from fabric_launcher import FabricLauncher

print("fabric-launcher installed successfully")

In [ ]:
# ============================================================================
# CELL 2 — Initialize launcher
# ============================================================================

launcher = FabricLauncher(
    notebookutils,
    environment="DEV",
    allow_non_empty_workspace=False,
    fix_zero_logical_ids=True,
    debug=False,
)

workspace_id = notebookutils.runtime.context.get("currentWorkspaceId", "")
print(f"Workspace ID: {workspace_id}")
print(f"Launcher ready")

In [ ]:
# ============================================================================
# CELL 3 — Download repo & deploy artifacts (staged)
# ============================================================================
# Stage 1: Lakehouses (must exist before notebooks that reference them)
# Stage 2: Notebooks (must exist before pipelines reference them)
# Stage 3: Pipelines
# Stage 4: Semantic Model + Data Agent (depend on lakehouse tables)
# ============================================================================

print("Downloading repo and deploying artifacts...")
print("This takes 3-5 minutes.\n")

downloader, deployer, report = launcher.download_and_deploy(
    repo_owner=GITHUB_OWNER,
    repo_name=GITHUB_REPO,
    branch=GITHUB_BRANCH,
    github_token=GITHUB_TOKEN if GITHUB_TOKEN else None,
    workspace_folder="workspace",
    item_type_stages=[
        ["Lakehouse"],                         # Stage 1
        ["Notebook"],                          # Stage 2
        ["DataPipeline"],                      # Stage 3
        ["SemanticModel", "DataAgent"],        # Stage 4
    ],
    validate_after_deployment=True,
    generate_report=True,
    deployment_retries=2,
)

print("\n" + "=" * 60)
print("  ARTIFACT DEPLOYMENT COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================================
# CELL 4 — Upload healthcare knowledge docs to Data Agent lakehouse
# ============================================================================

if UPLOAD_KNOWLEDGE_DOCS:
    import os

    # The repo was extracted by download_and_deploy; find the base path
    extract_base = downloader.extract_path  # e.g., .lakehouse/default/Files/src/<repo-folder>

    # Find the actual repo folder inside the extract path
    repo_dirs = [d for d in os.listdir(extract_base) if os.path.isdir(os.path.join(extract_base, d))]
    if repo_dirs:
        repo_root = os.path.join(extract_base, repo_dirs[0])
    else:
        repo_root = extract_base

    knowledge_src = os.path.join(repo_root, "healthcare_knowledge")

    if os.path.exists(knowledge_src):
        print("Uploading healthcare knowledge documents...")
        launcher.upload_files_to_lakehouse(
            lakehouse_name="lh_gold_curated",
            source_directory=knowledge_src,
            target_folder="healthcare_knowledge",
            file_patterns=["*.md"],
        )
        # Count uploaded files
        count = sum(len(files) for _, _, files in os.walk(knowledge_src) if files)
        print(f"Uploaded {count} knowledge documents to lh_gold_curated/Files/healthcare_knowledge/")
    else:
        print(f"Warning: healthcare_knowledge folder not found at {knowledge_src}")
else:
    print("Skipping knowledge doc upload (UPLOAD_KNOWLEDGE_DOCS=False)")

In [ ]:
# ============================================================================
# CELL 5 — Generate synthetic data
# ============================================================================

if GENERATE_DATA:
    print("Running NB_Generate_Sample_Data (generates ~10K patients, 100K encounters)...")
    print("This takes 2-4 minutes.\n")

    result = launcher.run_notebook_synchronous(
        notebook_name="NB_Generate_Sample_Data",
        parameters={},
        timeout_seconds=1200,  # 20 min max
    )

    if result.get("success"):
        print("\nData generation SUCCEEDED")
    else:
        status = result.get("status", "Unknown")
        print(f"\nData generation returned status: {status}")
        print("Check the NB_Generate_Sample_Data notebook for details.")
else:
    print("Skipping data generation (GENERATE_DATA=False)")

In [ ]:
# ============================================================================
# CELL 6 — Run the full-load pipeline
# ============================================================================

if RUN_PIPELINE:
    import requests, time, json

    token = notebookutils.credentials.getToken("pbi")
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    api_base = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

    # Find the pipeline ID
    print("Looking up PL_Healthcare_Master pipeline...")
    resp = requests.get(f"{api_base}/items?type=DataPipeline", headers=headers)
    resp.raise_for_status()
    pipelines = resp.json().get("value", [])
    pipeline = next((p for p in pipelines if p["displayName"] == "PL_Healthcare_Master"), None)

    if not pipeline:
        print("WARNING: PL_Healthcare_Master not found. Skipping pipeline run.")
        print("You can run it manually from the workspace.")
    else:
        pipeline_id = pipeline["id"]
        print(f"Pipeline ID: {pipeline_id}")

        # Trigger pipeline with load_mode=full
        print("Triggering pipeline with load_mode=full...")
        trigger_body = {
            "executionData": {
                "parameters": {
                    "load_mode": {"value": "full", "type": "string"}
                }
            }
        }
        resp = requests.post(
            f"{api_base}/items/{pipeline_id}/jobs/instances?jobType=Pipeline",
            headers=headers,
            json=trigger_body,
        )

        if resp.status_code in (200, 202):
            # Get job ID from Location header or response
            location = resp.headers.get("Location", "")
            print(f"Pipeline triggered. Polling for completion...")
            print(f"(This takes 8-15 minutes for full load)\n")

            # Poll until complete
            max_polls = 120  # 120 * 15s = 30 min max
            for poll in range(max_polls):
                time.sleep(15)
                try:
                    if location:
                        poll_resp = requests.get(location, headers=headers)
                    else:
                        poll_resp = requests.get(
                            f"{api_base}/items/{pipeline_id}/jobs/instances",
                            headers=headers,
                        )
                    if poll_resp.status_code == 200:
                        job_data = poll_resp.json()
                        status = job_data.get("status", "Unknown")
                        if status in ("Completed", "Succeeded"):
                            print(f"  Pipeline COMPLETED after {(poll+1)*15}s")
                            break
                        elif status in ("Failed", "Cancelled"):
                            print(f"  Pipeline {status} after {(poll+1)*15}s")
                            print(f"  Check the pipeline run history for details.")
                            break
                        else:
                            if poll % 4 == 0:  # Print every 60s
                                print(f"  [{(poll+1)*15}s] Status: {status}...")
                except Exception as e:
                    if poll % 4 == 0:
                        print(f"  [{(poll+1)*15}s] Polling... ({e})")
            else:
                print("  Pipeline still running after 30 min. Check workspace for status.")
        else:
            print(f"  Pipeline trigger returned {resp.status_code}: {resp.text}")
            print("  You can run it manually from the workspace.")
else:
    print("Skipping pipeline run (RUN_PIPELINE=False)")
    print("To run manually: Open PL_Healthcare_Master → Run with load_mode=full")

In [ ]:
# ============================================================================
# CELL 7 — Deploy Ontology via REST API (preview feature)
# ============================================================================
# Ontology is NOT supported by fabric-cicd, so we deploy via Fabric REST API.
# This creates entity types, relationships, and binds them to the semantic model.
# ============================================================================

if DEPLOY_ONTOLOGY:
    import requests, json, os

    token = notebookutils.credentials.getToken("pbi")
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    api_base = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

    # Find the ontology definition from the extracted repo
    repo_dirs = [d for d in os.listdir(downloader.extract_path) if os.path.isdir(os.path.join(downloader.extract_path, d))]
    repo_root = os.path.join(downloader.extract_path, repo_dirs[0]) if repo_dirs else downloader.extract_path
    ontology_dir = os.path.join(repo_root, "ontology", "Healthcare_Demo_Ontology_HLS")

    if os.path.exists(ontology_dir):
        print("Deploying Healthcare Ontology...")

        # Look for ontology definition files
        ontology_files = [f for f in os.listdir(ontology_dir) if f.endswith('.json') and f != '.platform']

        # Read the ontology definition
        ontology_def = None
        for fname in os.listdir(ontology_dir):
            fpath = os.path.join(ontology_dir, fname)
            if os.path.isdir(fpath):
                # Check for definition files inside subdirectories
                for sub_f in os.listdir(fpath):
                    if sub_f.endswith('.json'):
                        with open(os.path.join(fpath, sub_f), 'r') as f:
                            ontology_def = json.load(f)
                        break
            elif fname.endswith('.json') and fname != '.platform':
                with open(fpath, 'r') as f:
                    ontology_def = json.load(f)
                break

        if ontology_def:
            # Step 1: Create the ontology item
            create_body = {
                "displayName": "Healthcare_Demo_Ontology_HLS",
                "type": "Ontology",
            }
            resp = requests.post(f"{api_base}/items", headers=headers, json=create_body)
            if resp.status_code in (200, 201):
                ontology_id = resp.json()["id"]
                print(f"  Created ontology: {ontology_id}")

                # Step 2: Update with the definition
                # The ontology definition needs to reference the deployed semantic model
                # Find semantic model ID
                sm_resp = requests.get(f"{api_base}/items?type=SemanticModel", headers=headers)
                sm_list = sm_resp.json().get("value", []) if sm_resp.status_code == 200 else []
                sm = next((s for s in sm_list if s["displayName"] == "HealthcareDemoHLS"), None)

                if sm:
                    sm_id = sm["id"]
                    print(f"  Linked to semantic model: {sm_id}")
                    # Patch ontology definition with real workspace/semantic model IDs
                    def patch_ids(obj):
                        if isinstance(obj, dict):
                            for k, v in obj.items():
                                if k in ('workspaceId', 'workspace_id') and isinstance(v, str) and len(v) == 36:
                                    obj[k] = workspace_id
                                elif k in ('semanticModelId', 'artifactId', 'datasetId') and isinstance(v, str) and len(v) == 36:
                                    obj[k] = sm_id
                                else:
                                    patch_ids(v)
                        elif isinstance(obj, list):
                            for item in obj:
                                patch_ids(item)
                    patch_ids(ontology_def)

                    # Update ontology definition
                    update_resp = requests.post(
                        f"{api_base}/items/{ontology_id}/updateDefinition",
                        headers=headers,
                        json={"definition": {"parts": [{"path": "ontology-definition.json", "payload": json.dumps(ontology_def), "payloadType": "InlineBase64"}]}},
                    )
                    if update_resp.status_code in (200, 202):
                        print("  Ontology definition updated successfully")
                    else:
                        print(f"  Note: Ontology update returned {update_resp.status_code}")
                        print("  You may need to configure it manually in the Fabric UI.")
                else:
                    print("  Warning: SemanticModel not found. Ontology created but not linked.")
            elif resp.status_code == 409:
                print("  Ontology already exists — skipping")
            else:
                print(f"  Ontology creation returned {resp.status_code}: {resp.text[:200]}")
                print("  This is a preview feature — you may need to create it manually.")
        else:
            print("  No ontology definition JSON found. Skipping.")
    else:
        print(f"  Ontology folder not found at {ontology_dir}. Skipping.")
else:
    print("Skipping ontology deployment (DEPLOY_ONTOLOGY=False)")

In [ ]:
# ============================================================================
# CELL 8 — Deployment Summary
# ============================================================================

import requests

token = notebookutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}"}
api_base = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

resp = requests.get(f"{api_base}/items", headers=headers)
items = resp.json().get("value", []) if resp.status_code == 200 else []

# Group by type
by_type = {}
for item in items:
    t = item.get("type", "Unknown")
    by_type.setdefault(t, []).append(item["displayName"])

print("=" * 60)
print("  DEPLOYMENT COMPLETE")
print("=" * 60)
print(f"  Workspace: {workspace_id}")
print(f"  Total items: {len(items)}")
print()
for item_type in ["Lakehouse", "Notebook", "DataPipeline", "SemanticModel", "DataAgent", "Ontology", "SQLEndpoint"]:
    names = by_type.get(item_type, [])
    if names:
        print(f"  {item_type} ({len(names)}):")
        for n in sorted(names):
            print(f"    - {n}")

# Show any other types
shown = {"Lakehouse", "Notebook", "DataPipeline", "SemanticModel", "DataAgent", "Ontology", "SQLEndpoint"}
for item_type, names in by_type.items():
    if item_type not in shown:
        print(f"  {item_type} ({len(names)}):")
        for n in sorted(names):
            print(f"    - {n}")

print()
print("=" * 60)
print("  NEXT STEPS")
print("=" * 60)
print("  1. Open the HealthcareDemoHLS semantic model → verify tables loaded")
print("  2. Open the HealthcareHLSAgent → ask: 'What are the top denial reasons?'")
print("  3. For incremental loads: run NB_Generate_Incremental_Data, then")
print("     run PL_Healthcare_Master with load_mode=incremental")
print("=" * 60)